Installation and Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers==4.57.1 sentencepiece
!pip install sacremoses
from transformers import pipeline
translator = pipeline("translation_en_to_uk", model="Helsinki-NLP/opus-mt-en-uk")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 54.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/305M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Importing Training Libraries

In [ ]:
from transformers import MarianTokenizer, MarianMTModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch
import pandas as pd


Data loading...

In [ ]:
df = pd.read_csv("/content/tatoeba_full_tagged.csv")
src_texts = df["Tagged EN"].astype(str).tolist()
tgt_texts = df["ukr_reference"].astype(str).tolist()


tokenizer and model setup

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-uk"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

train_encodings = tokenizer(src_texts, text_target=tgt_texts,
                            truncation=True, padding=True, max_length=128)


dataset creation

In [ ]:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = TranslationDataset(train_encodings)


training args and training

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

training_args = Seq2SeqTrainingArguments(
    output_dir="./opusmt_gender_finetuned",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)
trainer.train()

/tmp/ipykernel_1347/2696055861.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
50,0.257700
100,0.130000
150,0.119100
200,0.080500
250,0.057000
300,0.071600
350,0.056200
400,0.039600
450,0.044300


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61586]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=483, training_loss=0.09184191735388082, metrics={'train_runtime': 73.8708, 'train_samples_per_second': 52.186, 'train_steps_per_second': 6.538, 'total_flos': 47983400386560.0, 'train_loss': 0.09184191735388082, 'epoch': 3.0})

Save and test

In [ ]:
#!zip -rv /content/drive/MyDrive/opusmt_gender_finetune.zip /content/opusmt_gender_finetuned

In [ ]:
trainer.save_model("./opusmt_gender_finetuned")

from transformers import pipeline
translator = pipeline("translation_en_to_uk", model="./opusmt_gender_finetuned")
translator("I am a good cleaner") #test




Device set to use cuda:0


[{'translation_text': 'Я добрий прибиральник.'}]

In [ ]:
import pandas as pd

df = pd.read_csv('/content/test_tagged.csv')

english_source = df['source'].tolist()
gt_translations = []

for i, english_text in enumerate(english_source, start=1):
    translated = translator(english_text)[0]['translation_text']
    gt_translations.append(translated)

# Add translations to dataframe
df['model_translation'] = gt_translations

# Save to CSV
df.to_csv("/content/drive/MyDrive/opusmt_tagged_translations.tsv", index=False, sep="\t")

print("Saved as /content/drive/MyDrive/opusmt_tagged_translations.tsv")


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved as /content/drive/MyDrive/opusmt_tagged_translations.tsv
